In [1]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
"""Spectrum 1D ON/OFF Analysis"""

'Spectrum 1D ON/OFF Analysis'

# Spectrum 1D ON/OFF Analysis with 3HWC Catalog


This notebook demonstrates how to:
1. Import necessary libraries and modules.
2. Load and select a source from the 3HWC catalog.
3. Configure an `AnalysisSpectrumConfig` object.
4. Perform spectrum analysis using `AnalysisSpectrum`.
5. Save and inspect the results, including flux points and fit results.

---


In [2]:
# Standard library imports
import astropy.units as u

# Third-party imports from gammapy
from gammapy.catalog import SourceCatalog3HWC
from gammapy.visualization import plot_spectrum_datasets_off_regions
from gammapy.data import Observation
from gammapy.maps import MapAxis
from gammapy.datasets import Datasets
from gammapy.utils.scripts import make_path
from gammapy.modeling.models import PowerLawSpectralModel, SkyModel, Models

# Third-party imports from regions
from regions import CircleSkyRegion

# feupy module imports
from feupy.catalog.hawc import get_flux_points_3hwc
from feupy.utils.coordinates import convert_skycoord_to_dict
from feupy.analysis.config import CTAOAnalysisConfig
from feupy.analysis.core import CTAOAnalysis
from feupy.utils.string_handling import string_to_filename_format
from feupy.visualization.counts import show_hist_counts


### 1. Load and Select Source
In this section, we load the 3HWC catalog and select a specific source (3HWC J1825-134) for analysis.

In [3]:
# Load the 3HWC catalog and select a source
catalog = SourceCatalog3HWC()
source = catalog["3HWC J1825-134"]

### 2. Configure `CTAOAnalysis`
Here, we configure the `CTAOAnalysisConfig` object with observation, dataset, and analysis settings for the spectrum analysis.

In [4]:
# Create and configure CTAOAnalysisConfig
config = CTAOAnalysisConfig()
# Define observation settings
position = source.position
# model_source = source.sky_model().copy(name="source")
config.observation.obs_cone = convert_skycoord_to_dict(position)
# config.observation.target.model = model_source.to_dict()

config.observation.position_angle = 0 * u.deg
config.observation.offset = 0.5 * u.deg
config.observation.livetime = 50 * u.h
config.observation.required_irfs = ["South", "AverageAz", "40deg", "50h"]

# Configure dataset settings
config.datasets.map_selection = ["edisp", "background", "exposure"]
config.datasets.safe_mask.methods = ["bkg-peak"]
config.datasets.safe_mask.parameters = {"aeff_percent": 10}
config.datasets.containment_correction = False
config.datasets.use_region_center = False
config.datasets.on_region = convert_skycoord_to_dict(position)
config.datasets.on_region.radius = 0.5 * u.deg

config.datasets.stack = False
config.datasets.on_off.acceptance = 1
config.datasets.on_off.acceptance_off = 5

# Configure energy axes
config.datasets.geom.axes.energy.min = 30 * u.GeV
config.datasets.geom.axes.energy.max = 300 * u.TeV
config.datasets.geom.axes.energy.nbins = 12
config.datasets.geom.axes.energy_true.min = 3 * u.GeV
config.datasets.geom.axes.energy_true.max = 500 * u.TeV
config.datasets.geom.axes.energy_true.nbins = 15

config.flux_points.energy.min = 30 * u.GeV
config.flux_points.energy.max = 300 * u.TeV
config.flux_points.energy.nbins = 12
config.flux_points.source = "source"

# Configure sensitivity settings
config.sensitivity.gamma_min = 5
config.sensitivity.n_sigma = 3
config.sensitivity.bkg_syst_fraction = 0.10

# Print configuration to verify
print(config)


CTAOAnalysisConfig

    general:
        log: {level: info, filename: null, filemode: null, format: null, datefmt: null}
        outdir: .
        n_jobs: 1
        datasets_file: null
        models_file: null
    observation:
        obs_cone: {frame: icrs, lon: 276.46 deg, lat: -13.4014 deg, radius: null}
        livetime: 50.0 h
        offset: 0.5 deg
        position_angle: 0.0 deg
        required_irfs: [South, AverageAz, 40deg, 50h]
    datasets:
        type: 1d
        stack: false
        geom:
            wcs:
                skydir: {frame: null, lon: null, lat: null}
                binsize: 0.02 deg
                width: {width: 5.0 deg, height: 5.0 deg}
                binsize_irf: 0.2 deg
            selection: {offset_max: 2.5 deg}
            axes:
                energy: {min: 30.0 GeV, max: 300.0 TeV, nbins: 12}
                energy_true: {min: 3.0 GeV, max: 500.0 TeV, nbins: 15}
        map_selection: [edisp, background, exposure]
        background:
          

### 3. Running Spectrum Analysis
In this section, we create an instance of `CTAOAnalysis` and perform the spectrum analysis based on the configured settings. The process includes simulating observations, running fits, and extracting flux points.

In [5]:
# Create and run spectrum analysis
analysis = CTAOAnalysis(config)

# Simulate observation and spectrum
analysis.simulate_observation()

INFO:feupy.analysis.config:Setting logging config: {'level': 'INFO', 'filename': None, 'filemode': None, 'format': None, 'datefmt': None}
INFO:feupy.analysis.core:Creating the pointing.
INFO:feupy.analysis.core:
 ON center:
<SkyCoord (ICRS): (ra, dec) in deg
    (276.46, -13.4014)>
INFO:feupy.analysis.core:
 Obsevation offset:
0.5 deg
INFO:feupy.analysis.core:
Pointing position:
<SkyCoord (ICRS): (ra, dec) in deg
    (276.46, -12.9014)>

/home/blinck-left/Coding/anaconda3/envs/gammapy-1.3/lib/python3.11/site-packages/gammapy/data/pointing.py:157: GammapyDeprecationWarning: Passing mode is deprecated and the argument will be removed in Gammapy 1.3. pointing mode is deduced from whether fixed_icrs or fixed_altaz is given
  warnings.warn(
INFO:feupy.analysis.core:
Pointing:
FixedPointingInfo:

mode:        PointingMode.POINTING
coordinates: <SkyCoord (ICRS): (ra, dec) in deg
    (276.46, -12.9014)>

INFO:feupy.analysis.core:
Setting observation parameters.
INFO:feupy.analysis.core:
irfs: 

In [6]:
model_simu =  PowerLawSpectralModel(
    index=3.0,
    amplitude=2.5e-12 * u.Unit("cm-2 s-1 TeV-1"),
    reference=1 * u.TeV,
)
model_source = SkyModel(spectral_model=model_simu, name="source")
analysis.get_spectrum_dataset(model_source)

INFO:feupy.analysis.core:Creating the background Maker.
INFO:feupy.analysis.core:Getting the observation.
INFO:feupy.analysis.core:
Observation

	obs id            : 0 
 	tstart            : 51544.00
	tstop             : 51546.08
	duration          : 180000.00 s
	pointing (icrs)   : 276.5 deg, -12.9 deg

	deadtime fraction : 0.0%


INFO:feupy.analysis.core:Getting the reference Dataset.
INFO:feupy.analysis.core:
reference: SpectrumDataset
---------------

  Name                            : 0 

  Total counts                    : 0 
  Total background counts         : 0.00
  Total excess counts             : 0.00

  Predicted counts                : 0.00
  Predicted background counts     : 0.00
  Predicted excess counts         : nan

  Exposure min                    : 0.00e+00 m2 s
  Exposure max                    : 0.00e+00 m2 s

  Number of total bins            : 12 
  Number of fit bins              : 0 

  Fit statistic type              : cash
  Fit statistic value (-2 log(L))

In [7]:
print(analysis.spectrum_dataset)

SpectrumDataset
---------------

  Name                            : 0 

  Total counts                    : 59190 
  Total background counts         : 11988.02
  Total excess counts             : 47201.98

  Predicted counts                : 59179.92
  Predicted background counts     : 11988.02
  Predicted excess counts         : 47191.90

  Exposure min                    : 1.34e+04 m2 s
  Exposure max                    : 4.32e+11 m2 s

  Number of total bins            : 12 
  Number of fit bins              : 10 

  Fit statistic type              : cash
  Fit statistic value (-2 log(L)) : -1056106.11

  Number of models                : 1 
  Number of parameters            : 3
  Number of free parameters       : 2

  Component 0: SkyModel
  
    Name                      : source
    Datasets names            : None
    Spectral model type       : PowerLawSpectralModel
    Spatial  model type       : 
    Temporal model type       : 
    Parameters:
      index                   

In [8]:
analysis.get_datasets()

name,counts,excess,sqrt_ts,background,npred,npred_background,npred_signal,exposure_min,exposure_max,livetime,ontime,counts_rate,background_rate,excess_rate,n_bins,n_fit_bins,stat_type,stat_sum,counts_off,acceptance,acceptance_off,alpha
,,,,,,,,m2 s,m2 s,s,s,1 / s,1 / s,1 / s,,,,,,,,
str5,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,str5,float64,int64,float64,float64,float64
obs-0,59378,47384.8,263.0691150485444,11993.199999999999,19890.666666666668,19890.666666666668,nan,13355.569479766584,431761766439.2976,180000.0,180000.0,0.32987777777777777,0.06662888888888888,0.2632488888888889,12,10,wstat,72415.85741194343,59966,10.0,50.00000000000001,0.19999999999999998


In [9]:
print(analysis.datasets)

Datasets
--------

Dataset 0: 

  Type       : SpectrumDatasetOnOff
  Name       : obs-0
  Instrument : CTA
  Models     : 




In [10]:
analysis.datasets.info_table()

name,counts,excess,sqrt_ts,background,npred,npred_background,npred_signal,exposure_min,exposure_max,livetime,ontime,counts_rate,background_rate,excess_rate,n_bins,n_fit_bins,stat_type,stat_sum,counts_off,acceptance,acceptance_off,alpha
,,,,,,,,m2 s,m2 s,s,s,1 / s,1 / s,1 / s,,,,,,,,
str5,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,str5,float64,int64,float64,float64,float64
obs-0,59378,47384.8,263.0691150485444,11993.199999999999,19890.666666666668,19890.666666666668,nan,13355.569479766584,431761766439.2976,180000.0,180000.0,0.32987777777777777,0.06662888888888888,0.2632488888888889,12,10,wstat,72415.85741194343,59966,10.0,50.00000000000001,0.19999999999999998


In [11]:
analysis.set_models(Models(model_source))

INFO:feupy.analysis.core:Reading model.
INFO:feupy.analysis.core:Models

Component 0: SkyModel

  Name                      : source
  Datasets names            : None
  Spectral model type       : PowerLawSpectralModel
  Spatial  model type       : 
  Temporal model type       : 
  Parameters:
    index                         :      3.000   +/-    0.00             
    amplitude                     :   2.50e-12   +/- 0.0e+00 1 / (TeV s cm2)
    reference             (frozen):      1.000       TeV         




In [12]:
analysis.run_fit()

INFO:feupy.analysis.core:Fitting datasets.
INFO:feupy.analysis.core:OptimizeResult

	backend    : minuit
	method     : migrad
	success    : True
	message    : Optimization terminated successfully.
	nfev       : 43
	total stat : 10.73

CovarianceResult

	backend    : minuit
	method     : hesse
	success    : True
	message    : Hesse terminated successfully.



In [13]:
analysis.fit_result.parameters.to_table()

type,name,value,unit,error,min,max,frozen,link,prior
str1,str9,float64,str14,float64,float64,float64,bool,str1,str1
,index,2.9917e+00,,6.609e-03,nan,nan,False,,
,amplitude,2.5491e-12,TeV-1 s-1 cm-2,3.298e-14,nan,nan,False,,
,reference,1.0000e+00,TeV,0.000e+00,nan,nan,True,,


In [14]:
analysis.get_flux_points()

INFO:feupy.analysis.core:Calculating flux points.
INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
INFO:feupy.analysis.core:
      e_ref                 dnde          ...      sqrt_ts      
       GeV            1 / (TeV s cm2)     ...                   
------------------ ---------------------- ... ------------------
 44.03397802866209                    nan ...                nan
 94.86832980505132 2.9350219677298276e-09 ...  184.9928097607481
204.38762071738844 2.9293759288072797e-10 ... 146.05331043250837
  440.339780286621  2.933786767406246e-11 ...  96.65336011132577
 948.6832980505134 3.1056521977203793e-12 ...  72.94425936048307
2043.8762071738831  3.060315632685498e-13 ...  40.30987521292156
 4403.397802866211  2.907597916729892e-14 ... 20.468087416421923
 9486.832980505136 2.0154635579883104e-15 ...  9.149433205548888
20438.762071738816 2.8586753707065515e-16 ...  7.083026552095542
 44033.97802866208 2.3991195330254436e-17 ...  3.786036945552148
  94868.329805051

In [15]:
analysis.flux_points

In [16]:
# Run the sensitivity analysis
analysis.compute_sensitivity()

# Write the sensitivity table to a file (CSV in this case)
analysis.write_table_sensitivity()

# Read the sensitivity table back if needed
sensitivity_table = analysis.read_table_sensitivity()

# Print the sensitivity table summary
print("\nSensitivity Table:")
print(sensitivity_table)

INFO:feupy.analysis.core:
Creating ON/OFF Dataset:
INFO:feupy.analysis.core:
ON/OFF Dataset:
SpectrumDatasetOnOff
--------------------

  Name                            : YdOCN6_p 

  Total counts                    : 59522 
  Total background counts         : 12005.80
  Total excess counts             : 47516.20

  Predicted counts                : 59401.04
  Predicted background counts     : 12012.62
  Predicted excess counts         : 47388.41

  Exposure min                    : 1.34e+04 m2 s
  Exposure max                    : 4.32e+11 m2 s

  Number of total bins            : 12 
  Number of fit bins              : 10 

  Fit statistic type              : wstat
  Fit statistic value (-2 log(L)) : 14.27

  Number of models                : 1 
  Number of parameters            : 3
  Number of free parameters       : 2

  Component 0: SkyModel
  
    Name                      : source
    Datasets names            : None
    Spectral model type       : PowerLawSpectralModel
    Spa

FileNotFoundError: [Errno 2] No such file or directory: '/home/blinck-left/Coding/GitHub/feupy/examples/None/sens_CTAO-South_40deg_50h_livetime50.0h.fits'

In [ ]:
analysis.table_sens

In [ ]:
table = analysis.datasets.info_table()

In [ ]:
show_hist_counts(table)

In [ ]:
analysis.flux_points.data.to_table()

### 4. Inspect Results
Here, we review the results of the spectrum analysis, including the fit parameters and computed flux points.

In [ ]:
# Display the fit parameters
print("Fit Parameters:")
print(analysis.fit_result)

# Display the flux points
print("Flux Points:")
print(analysis.flux_points)

## 5. Summary

In this notebook, we:

1. Selected a source from the 3HWC catalog.
2. Configured and ran a 1D ON/OFF spectrum analysis using `AnalysisSpectrum`.
3. Printed and saved the fit results and flux points for further analysis.

This modular design
